# Plan C: small object detection with multi-scale fusion and attention

Run the cells from top to bottom. Read `plans/03-small-object.md` first. Before you start:

1. Use **Runtime > Change runtime type** and pick a GPU (for example T4).
2. Nothing to upload: Ultralytics downloads and converts VisDrone itself (about 2 GB; allow about 4 GB of free disk). VisDrone is for educational use here; cite Zhu et al., TPAMI 2021, and do not republish it.
3. Edit the **Configuration** cell: the variant, the seed and the number of epochs.

This is the heaviest task for free Colab. Explore with `yolo26n`, 640 px, few epochs and one seed; keep 1280 px for the baseline and the final model.

## 1. Install and check the GPU

In [ ]:
# Clone the project (or update it when the runtime is reused), install the pinned requirements and show the GPU
import os
import subprocess
import sys
from pathlib import Path
# Address of the team repository
repositoryUrl = "https://github.com/keithku321/object-detection"
# Folder on the Colab machine that holds the clone
repositoryFolder = Path("/content/objectDetection")
# A private repository needs a GitHub token stored as the Colab Secret GITHUB_TOKEN (key icon in the left bar); never paste a token into a cell
try:
    from google.colab import userdata
    githubToken = userdata.get("GITHUB_TOKEN")
except Exception:
    githubToken = None
cloneUrl = repositoryUrl.replace("https://", f"https://{githubToken}@") if githubToken else repositoryUrl
gitCommand = ["git", "-C", str(repositoryFolder), "pull", "--quiet"] if repositoryFolder.exists() else ["git", "clone", "--quiet", cloneUrl, str(repositoryFolder)]
gitResult = subprocess.run(gitCommand, capture_output=True, text=True)
# Print git's message with the token removed, so the token can never end up in a saved notebook
if gitResult.returncode != 0: print(gitResult.stderr.replace(githubToken or "GITHUB_TOKEN", "***"))
os.chdir(repositoryFolder)
if str(repositoryFolder) not in sys.path: sys.path.insert(0, str(repositoryFolder))
# Colab already has PyTorch; requirements.txt adds the pinned Ultralytics and the task libraries without touching it
!pip install -q -r requirements.txt
# Show the attached GPU; if this prints an error then change the runtime type to a GPU
!nvidia-smi
from common.environment import printEnvironment
printEnvironment()

## 2. Configuration

In [ ]:
# Name of this task; it names the results file and the run folders
taskName = "smallobject"
# Dataset YAML shipped with Ultralytics; it downloads and converts VisDrone on first use
datasetYamlName = "VisDrone.yaml"
# Variant to train: features joined by "+", from p2, cbam, eca, simam, ca, dysample, wfusion, nwd (see smallobject/train.py)
variantName = "p2"
# Model scale; n is nano, s is small
modelScale = "n"
# Seed; exploration uses 0, the final runs use 0, 1 and 2
seed = 0
# Number of full passes over the training data; time one epoch first and set this from the compute budget (gate G1)
trainingEpochs = 50
# Image size in pixels; 640 for exploration, 1280 only for the baseline and the final model
imageSize = 640
# Images per batch; lower it to 8 or 4 at 1280 px
batchSize = 16
# Weight of the NWD term in the box loss, used only by variants that contain "nwd"; the plan tries 0.3 and 0.5
nwdWeight = 0.5
# Drive folder for results, so that weights survive when Colab resets the runtime
driveResultsFolder = Path("/content/drive/MyDrive/yoloProjectResults")

## 3. Mount Google Drive and unpack the dataset

Drive holds results and checkpoints. The dataset is downloaded by Ultralytics to the local disk of the Colab machine.

In [ ]:
from google.colab import drive
# Mount Google Drive; a pop-up will ask for permission. Drive holds results and checkpoints only, never the working copy of the dataset
drive.mount("/content/drive")
driveResultsFolder.mkdir(parents=True, exist_ok=True)
# Bring back results rows of earlier sessions: a fresh clone only contains the rows that were committed
from common.results import syncResultFiles
print(syncResultFiles(taskName, driveResultsFolder / "results"), "results rows so far")

from common.datasets import getManifestEntry
from smallobject.evalCoco import resolveSplitImages
print(getManifestEntry("visdrone")["license"])
# Resolving a split makes Ultralytics download and convert VisDrone when it is not on this machine yet
trainImagePaths, classNames = resolveSplitImages(datasetYamlName, "train")
valImagePaths, _ = resolveSplitImages(datasetYamlName, "val")
print(len(trainImagePaths), "training images,", len(valImagePaths), "validation images,", len(classNames), "classes")

## 4. Helper functions

The logic lives in `common/` and `smallobject/` so it can be tested on a PC; this cell only imports it and adds two display helpers.

In [ ]:
from IPython.display import display
from PIL import Image
# File endings that count as images
imageSuffixes = {".jpg", ".jpeg", ".png"}

# Display the first few images from a folder inside the notebook
def showSavedImages(folderPath, maximumCount=4):
    imageFiles = [file for file in sorted(Path(folderPath).iterdir()) if file.suffix.lower() in imageSuffixes]
    for imageFile in imageFiles[:maximumCount]:
        display(Image.open(imageFile))

# Save the results rows to Drive and print the summary table (mean ± std over seeds)
def saveAndShowResults():
    from common.results import syncResultFiles
    from common.summariseResults import writeSummary
    syncResultFiles(taskName, driveResultsFolder / "results")
    print(writeSummary() or "No results rows yet.")

from smallobject.evalCoco import countObjectsPerImage, endToEndDetectionCap
from smallobject.nwdLoss import computeNwdConstant
from smallobject.train import runVariant

## 5. Create dataset.yaml and sanity-check the data

The dataset YAML comes with Ultralytics, so this section only checks the data. Two numbers matter for this task:

- **Objects per image.** The NMS-free head of YOLO26 returns at most 300 detections per image unless the limit is raised; the evaluation here raises it to 500. The statistics show how many images would be affected.
- **Typical object size**, which becomes the constant `C` of the NWD loss.

In [ ]:
# Objects per image: images above the cap lose detections when max_det stays at its default
for splitName, imagePaths in (("train", trainImagePaths), ("val", valImagePaths)):
    print(splitName, countObjectsPerImage(imagePaths), f"(cap = {endToEndDetectionCap})")
# Mean object size in pixels at the training image size, used as the NWD constant
nwdConstant = computeNwdConstant(trainImagePaths, imageSize)
print(f"Typical object size at {imageSize} px: {nwdConstant:.1f} px")

## 6. Baseline

Train the unmodified `yolo26` model first (experiment S0). YOLO26 already includes small-target-aware label assignment, so this baseline is stronger than the YOLOv8 baseline that older papers compare against. Every variant uses the same image size, epochs and seed. A finished baseline is skipped.

In [ ]:
# Train the plain pretrained model, then evaluate COCO-style AP by object size and log one row
baselineWeightsPath = runVariant("baseline", seed, trainingEpochs, driveResultsFolder, datasetYamlName, modelScale, imageSize, batchSize)
saveAndShowResults()

## 7. Train

Add one idea at a time: `p2`, then an attention block, `dysample`, `wfusion`, `nwd`, and finally the best combination. Results are written straight to Drive so the weights survive a disconnect.

Read the log line **"Transferred N/M items from pretrained weights"**. The P2 head has no pretrained weights (only the backbone and the first neck layers transfer), and every new layer starts untrained, so compare with a baseline trained for the same number of epochs.

In [ ]:
# Train the variant chosen in the configuration cell, evaluate AP by object size and log one row in results/smallobject.csv
variantWeightsPath = runVariant(variantName, seed, trainingEpochs, driveResultsFolder, datasetYamlName, modelScale, imageSize, batchSize, nwdWeight=nwdWeight, nwdConstant=nwdConstant)
saveAndShowResults()

### If Colab disconnected during training

Run the earlier setup cells (1 to 5) again, then run the cell below. It is the same call as the training cell: every run has a fixed folder on Drive, so training continues from the last saved epoch, and finished runs are skipped.

In [ ]:
# Continue the interrupted run from its last checkpoint on Drive (same call as above; finished runs are skipped)
# variantWeightsPath = runVariant(variantName, seed, trainingEpochs, driveResultsFolder, datasetYamlName, modelScale, imageSize, batchSize, nwdWeight=nwdWeight, nwdConstant=nwdConstant)
# saveAndShowResults()

## 8. Evaluate the trained model

Accuracy rows (mAP and AP by size) are already logged. This cell adds the speed protocol, because a small-object gain only counts if the model stays lightweight: compare parameters, GFLOPs and latency with the baseline.

In [ ]:
from common.benchmark import benchmarkModel
from common.results import appendResultRow, buildResultRow
# Speed and size with the shared protocol, for the baseline and the variant
for modelLabel, weightsPath in (("baseline", baselineWeightsPath), (variantName, variantWeightsPath)):
    speedMetrics = benchmarkModel(weightsPath)
    print(modelLabel, speedMetrics)
    appendResultRow(buildResultRow(taskName, f"yolo26{modelScale}", modelLabel, seed, {key: value for key, value in speedMetrics.items() if isinstance(value, (int, float))}, notes="speed protocol"))
saveAndShowResults()

## 9. Look at real predictions

Scores hide failures, so look at the images, especially crowded scenes and distant pedestrians.

In [ ]:
from ultralytics import YOLO
from common.registerModules import registerCustomModules
from common.training import endToEndArguments
registerCustomModules()
# Predict on a few validation images with the trained model and save annotated copies; max_det is raised for dense scenes
trainedModel = YOLO(str(variantWeightsPath))
predictionResults = trainedModel.predict(source=[str(path) for path in valImagePaths[:8]], save=True, conf=0.25, max_det=500, imgsz=imageSize, project="/content/runs/trained", name="predict", exist_ok=True, **endToEndArguments)
showSavedImages("/content/runs/trained/predict", maximumCount=8)

## 10. AP by object size

This is the cell that turns the project into a finding: does the change help **small** objects (area below 32 x 32 pixels in the original image) without hurting medium and large ones? `cocoAgreesWithUltralytics` must be 1; if it is 0 the label conversion is wrong.

In [ ]:
from common.results import readResultRows, resultsFolder
# Print AP by object size for every variant logged so far
for row in readResultRows(resultsFolder / "smallobject.csv"):
    if not row.get("apSmall"): continue
    print(f"{row['variant']:<32} seed {row['seed']}  AP_s = {float(row['apSmall']):.3f}  AP_m = {float(row['apMedium']):.3f}  AP_l = {float(row['apLarge']):.3f}  mAP50-95 = {float(row['map50_95']):.3f}  agrees = {row['cocoAgreesWithUltralytics']}")

## 11. Export and download (optional)

Download the weights for the demo app (`weights/` folder of the repository). Export only if you need another format.
The results rows now sit in `results/smallobject.csv` inside the clone and in `yoloProjectResults/results/` on Drive. Rows are part of the repository: download the CSV (or copy it from Drive) and commit it on your branch, without weights. Strip the outputs of this notebook before committing it (**Edit > Clear all outputs**).

In [ ]:
from google.colab import files
# Download the final weights to your own computer; the file name follows <task>-<variant>-seed<k>.pt
files.download(str(variantWeightsPath))
# Export to ONNX for the deployment number; nms=False exports the NMS-free head
exportedModelPath = trainedModel.export(format="onnx", imgsz=imageSize, **endToEndArguments)
print(exportedModelPath)
saveAndShowResults()